### 练习1：诊断一个失败的PINN

In [2]:
import torch
import torch.nn as nn

# 下面这个PINN训练失败，请找出问题并修复
broken_model = nn.Sequential(
    nn.Linear(2, 20),
    nn.ReLU(),
    nn.Linear(20, 20),
    nn.ReLU(),
    nn.Linear(20, 1)
)

# 提示1：检查激活函数
# 提示2：检查网络宽度
# 提示3：检查...

# 修复后：
fixed_model = nn.Sequential(
    nn.Linear(2, 50),
    nn.Tanh(),
    nn.Linear(50, 50),
    nn.Tanh(),
    nn.Linear(50, 50),
    nn.Tanh(),
    nn.Linear(50, 1)
)

### 练习2：实现自适应权重

In [4]:
class AdaptivePINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 50), nn.Tanh(),
            nn.Linear(50, 50), nn.Tanh(),
            nn.Linear(50, 1)
        )
        # 可学习的权重参数
        self.log_vars = nn.Parameter(torch.zeros(3))

    def forward(self, x):
        return self.net(x)

    def loss(self, l_data, l_pde, l_bc):
        w = torch.softmax(-self.log_vars, dim=0)
        total = w[0]*l_data + w[1]*l_pde + w[2]*l_bc + self.log_vars.sum()
        return total

def compute_individual_losses(model):
    """计算三项独立损失（以一个简单的热传导方程为例）"""
    # 数据点：已知观测值
    x_data = torch.rand(100, 2)  # (x, t) 随机采样
    u_true = torch.sin(torch.pi * x_data[:, 0:1]) * torch.exp(-x_data[:, 1:2])  # 解析解
    u_pred_data = model(x_data)
    loss_data = ((u_pred_data - u_true) ** 2).mean()

    # PDE残差点：∂u/∂t = α·∂²u/∂x²（热传导方程）
    x_pde = torch.rand(500, 2, requires_grad=True)
    u_pred_pde = model(x_pde)
    u_t = torch.autograd.grad(u_pred_pde.sum(), x_pde, create_graph=True)[0][:, 1:2]
    u_x = torch.autograd.grad(u_pred_pde.sum(), x_pde, create_graph=True)[0][:, 0:1]
    u_xx = torch.autograd.grad(u_x.sum(), x_pde, create_graph=True)[0][:, 0:1]
    alpha = 1.0 / (torch.pi ** 2)  # 热扩散系数，使 sin(πx)·exp(-t) 满足热传导方程
    pde_residual = u_t - alpha * u_xx
    loss_pde = (pde_residual ** 2).mean()

    # 边界点：x=0 和 x=1
    x_bc = torch.cat([
        torch.zeros(50, 1),  # x=0
        torch.ones(50, 1)    # x=1
    ], dim=0)
    t_bc = torch.rand(100, 1)
    x_bc_full = torch.cat([x_bc, t_bc], dim=1)
    u_pred_bc = model(x_bc_full)
    u_true_bc = torch.sin(torch.pi * x_bc[:, 0:1]) * torch.exp(-t_bc)  # sin(0)=sin(π)=0，零边界
    loss_bc = ((u_pred_bc - u_true_bc) ** 2).mean()

    return loss_data, loss_pde, loss_bc

# 训练时同时优化网络参数和权重参数
model = AdaptivePINN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5000):
    optimizer.zero_grad()
    ld, lp, lb = compute_individual_losses(model)
    total = model.loss(ld, lp, lb)
    total.backward()
    optimizer.step()

    if epoch % 1000 == 0:
        w = torch.softmax(-model.log_vars, dim=0).detach()
        print(f"Epoch {epoch}: weights = {w.tolist()}")

Epoch 0: weights = [0.3333333432674408, 0.3333333432674408, 0.3333333432674408]
Epoch 1000: weights = [0.3345929980278015, 0.33216798305511475, 0.33323904871940613]
Epoch 2000: weights = [0.33486613631248474, 0.3319172263145447, 0.33321666717529297]
Epoch 3000: weights = [0.33495190739631653, 0.3318502604961395, 0.33319780230522156]
Epoch 4000: weights = [0.33500444889068604, 0.33182406425476074, 0.3331714868545532]
